# 1. Completed retrospective rainfall benchmark

This companion reads **completed saved results** for protocol
`revised_archive_safe_label_embargo_raw_predictions_v1`; it never computes model
updates, source extraction, checkpoint selection, or new forecasts. Missing or
inconsistent artifacts stop the notebook. Change the explicit path below to
inspect a completed run.

The task is monthly CHIRPS v2 regional rainfall at the first day of the target
month, with 12 historical origin rows. SAFE target roles per client are training
1983-01–2007-11 (299), inner monitor 2008-01–2010-11 (35), outer validation
2011-01–2016-11 (71), and retrospective evaluation 2017-01–2024-12 (96).
The three boundary December labels are embargoed. Five paired seeds are
42, 123, 2024, 7, 999. Present-day revised archives and latency assumptions do
not reconstruct historical source vintages; this is not an untouched prospective
test. Regions 1 and 5 overlap. Raw negative predictions are retained. Global,
personal and fixed-anchor component identities remain separate.

All model names and plot settings are explicit inspection choices, not another
selection procedure. Accuracy plots use saved per-seed results; no prediction
ensemble is silently scored.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Launch from the repository root or its notebooks directory.
REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / 'caadapfl').is_dir() and (REPOSITORY_ROOT.parent / 'caadapfl').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
RUN_DIR = REPOSITORY_ROOT / 'runs' / 'revision_v2'
VIEW_MODELS = (
    'CAAA_GLOBAL_FULL', 'FEDAVG_GLOBAL_FULL', 'FEDPROX_GLOBAL_FULL',
    'CENTRAL_GLOBAL_FULL', 'LOCAL_PERSONAL_FULL', 'DITTO_PERSONAL_FULL',
    'GRAPH_ORIGINAL_PERSONAL_FULL', 'GRAPH_ENRICHED_PERSONAL_FULL',
    'GRAPH_UNIFORM_PERSONAL_FULL', 'raw_pooled_ET_FULL_latest',
    'client_normalized_pooled_ET_FULL_latest',
)
PAIR = ('CAAA_GLOBAL_FULL', 'FEDAVG_GLOBAL_FULL')
PLOT_CLIENT = 'Client1_NW_Arid'
PLOT_SEED = 42
PLOT_START = '2023-01'


## 2. Verify the completed run before reading tables

The self-contained reader checks run/protocol/schedule digests, frozen choices,
all admitted fit receipts and artifact hashes, the matching repository modules,
the common 480 target keys per seed/model, and metric/paired-point arithmetic.
It deserializes no pickle or tensor model. SHA256 verifies consistency with saved
receipts; it is not an author signature or a secure-aggregation mechanism.
Hashing checkpoint bytes can take time. No incomplete run is displayed as an
accepted scientific result.

In [ ]:
"""Read completed, integrity-checked rainfall results without model execution.

Only JSON/CSV/text bytes are read. No pickle, tensor state, model import or
subprocess is used. SHA256s establish consistency with the saved receipts;
they are not signatures or a guarantee that an untrusted author is truthful.
"""
from __future__ import annotations
from pathlib import Path, PurePosixPath, PureWindowsPath
import hashlib
import json
import re

import numpy as np
import pandas as pd

SEEDS=(42,123,2024,7,999)
CLIENTS=('Client1_NW_Arid','Client2_Indo_Gangetic','Client3_NE_Himalaya',
         'Client4_South_Peninsular','Client5_Central_India')
MONTHS=tuple(str(m) for m in pd.period_range('2017-01','2024-12',freq='M'))
REQUIRED_CSV=('raw_predictions','per_client','aggregate_per_seed','seed_summary',
              'paired_block_bootstrap_summary','paired_per_seed_summary',
              'negative_forecast_diagnostics','fit_timing','fit_cost_ledger')
REQUIRED_JSON=('model_identities','hardware','metrics_receipt','fit_cost_receipt',
               'shared_graph_setup_ledger','PR_curves')

class ResultIntegrityError(ValueError):
    """Incomplete or inconsistent saved run; do not display it as accepted."""

def require(condition,message):
    if not condition:raise ResultIntegrityError(message)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as fp:
        for block in iter(lambda:fp.read(1<<20),b''):h.update(block)
    return h.hexdigest()

def canonical_digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def safe_child(root,relative):
    """Accept saved Windows/POSIX relative paths; reject escapes and symlinks."""
    require(isinstance(relative,str) and bool(relative),'Empty/invalid receipt path')
    normalized=relative.replace('\\','/')
    parts=PurePosixPath(normalized)
    require(not parts.is_absolute() and not PureWindowsPath(relative).drive,
            f'Absolute receipt path is not portable: {relative}')
    require('..' not in parts.parts,f'Receipt path escapes its directory: {relative}')
    resolved=(Path(root)/Path(*parts.parts)).resolve()
    require(resolved.is_relative_to(Path(root).resolve()),f'Receipt path escapes through a symlink: {relative}')
    return resolved

def read_json(path):
    try:return json.loads(Path(path).read_text(encoding='utf-8'))
    except (OSError,json.JSONDecodeError) as error:
        raise ResultIntegrityError(f'Cannot read saved JSON {path}: {error}') from error

def check_hash(path,expected):
    require(isinstance(expected,str) and re.fullmatch('[0-9a-f]{64}',expected) is not None,
            f'Invalid SHA256 in receipt for {path}')
    require(Path(path).is_file(),f'Missing saved artifact: {path}')
    require(sha256(path)==expected,f'Artifact checksum differs: {path}')

def verify_files(root,files):
    require(isinstance(files,dict) and bool(files),'Missing artifact hash map')
    total=0
    for relative,expected in files.items():
        path=safe_child(root,relative);check_hash(path,expected);total+=path.stat().st_size
    return len(files),total

def without_runtime_root(value):
    if isinstance(value,dict):return {k:without_runtime_root(v) for k,v in value.items() if k!='root'}
    if isinstance(value,list):return [without_runtime_root(v) for v in value]
    return value

def regression(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    denominator=float(np.sum((y-y.mean())**2))
    require(denominator>0,'Constant saved observations make R² undefined')
    return 1-float(np.sum((y-p)**2))/denominator,float(np.sqrt(np.mean((y-p)**2)))

def close(actual,expected,label):
    require(np.allclose(actual,expected,rtol=1e-8,atol=1e-9,equal_nan=False),
            f'Saved metric does not reconcile with raw predictions: {label}')

def validate_tables(frames,metadata):
    raw=frames['raw_predictions']
    keys=['model','seed','client','target_month']
    require(set(keys+['y','prediction']).issubset(raw),'Missing keyed raw prediction columns')
    require(not raw.duplicated(keys).any(),'Duplicate raw prediction keys')
    require(np.isfinite(raw[['y','prediction']].to_numpy(float)).all(),'Nonfinite saved predictions/observations')
    require((raw.y>=0).all(),'Negative observed rainfall')
    require(set(raw.client)==set(CLIENTS),'Saved study client cohort differs')
    require(set(raw.seed.astype(int))==set(SEEDS),'Saved paired seed cohort differs')
    require(sorted(raw.target_month.unique())==list(MONTHS),'Saved evaluation calendar differs')
    expected=pd.MultiIndex.from_product([sorted(CLIENTS),MONTHS],names=['client','target_month'])
    expected_rows=len(CLIENTS)*len(MONTHS)
    truth=None
    calculated=[]
    per=frames['per_client'].set_index(['model','seed','client'],verify_integrity=True)
    aggregate=frames['aggregate_per_seed'].set_index(['model','seed','scope'],verify_integrity=True)
    for (model,seed),part in raw.groupby(['model','seed'],sort=True):
        part=part.sort_values(['client','target_month']).set_index(['client','target_month'])
        require(len(part)==expected_rows and part.index.equals(expected),f'Incomplete common target keys: {model}/{seed}')
        y=part.y.to_numpy(float)
        if truth is None:truth=y
        require(np.array_equal(y,truth),'Observed values differ between methods or seeds')
        client_scores=[]
        for client,group in part.groupby(level='client',sort=True):
            score=regression(group.y,group.prediction);client_scores.append(score)
            require((model,seed,client) in per.index,f'Missing saved client metric: {model}/{seed}/{client}')
            saved=per.loc[(model,seed,client)]
            require(int(saved['n'])==len(MONTHS),'Per-client metric count differs')
            close(saved[['R2','RMSE_mm']].to_numpy(float),score,f'{model}/{seed}/{client}')
        macro=np.asarray(client_scores).mean(axis=0);pooled=regression(part.y,part.prediction)
        for scope,score in [('macro_equal_client',macro),('pooled_all_client_months',pooled)]:
            require((model,seed,scope) in aggregate.index,f'Missing aggregate scope: {model}/{seed}/{scope}')
            close(aggregate.loc[(model,seed,scope),['R2','RMSE_mm']].to_numpy(float),score,f'{model}/{seed}/{scope}')
        calculated.append(dict(model=model,seed=int(seed),R2=float(macro[0]),RMSE_mm=float(macro[1])))
    calc=pd.DataFrame(calculated)
    require(set(calc.model)==set(frames['seed_summary'].model),'Seed-summary method labels differ')
    summary=frames['seed_summary'].set_index('model',verify_integrity=True)
    for model,group in calc.groupby('model'):
        require(len(group)==len(SEEDS),f'Missing saved seed for {model}')
        for metric in ('R2','RMSE_mm'):
            close(summary.loc[model,[metric+'_mean',metric+'_std']].to_numpy(float),
                  [group[metric].mean(),group[metric].std(ddof=1)],f'{model}/{metric}/seed summary')
    require(len(raw)==int(metadata['evaluation']['raw_predictions']),'Evaluation receipt row count differs')
    require(raw.model.nunique()==int(metadata['evaluation']['models']),'Evaluation receipt model count differs')
    require(len(per)==len(calc)*len(CLIENTS),'Unexpected per-client metric rows')
    require(len(aggregate)==len(calc)*2,'Unexpected aggregate metric rows')
    comparisons=[tuple(p) for p in metadata['protocol']['comparisons']]
    intervals=frames['paired_block_bootstrap_summary']
    require(not intervals.duplicated(['model_a','model_b','metric','block_length_months']).any(),'Duplicate paired interval rows')
    require(set(zip(intervals.model_a,intervals.model_b))==set(comparisons),'Interval comparison family differs from frozen protocol')
    means=calc.groupby('model')[['R2','RMSE_mm']].mean()
    expected_interval_keys={(a,b,m,l) for a,b in comparisons for m in ('macro_R2','macro_RMSE_mm') for l in (12,6,24)}
    actual_interval_keys=set(map(tuple,intervals[['model_a','model_b','metric','block_length_months']].to_numpy()))
    require(actual_interval_keys==expected_interval_keys,'Missing primary/sensitivity interval rows')
    require(np.isfinite(intervals[['delta_a_minus_b','CI95_low','CI95_high']].to_numpy(float)).all(),'Nonfinite interval')
    require((intervals.CI95_low<=intervals.CI95_high).all(),'Reversed interval endpoints')
    for row in intervals.itertuples():
        metric='R2' if row.metric=='macro_R2' else 'RMSE_mm'
        close(row.delta_a_minus_b,means.loc[row.model_a,metric]-means.loc[row.model_b,metric],f'paired interval point/{row.model_a}/{row.model_b}/{metric}')
    paired=frames['paired_per_seed_summary']
    require(not paired.duplicated(['model_a','model_b','seed']).any(),'Duplicate paired seed differences')
    expected_paired={(a,b,s) for a,b in comparisons for s in SEEDS}
    require(set(map(tuple,paired[['model_a','model_b','seed']].to_numpy()))==expected_paired,'Paired seed family differs')
    lookup=calc.set_index(['model','seed'])
    for row in paired.itertuples():
        for metric,col in [('R2','delta_macro_R2_a_minus_b'),('RMSE_mm','delta_macro_RMSE_mm_a_minus_b')]:
            close(getattr(row,col),lookup.loc[(row.model_a,row.seed),metric]-lookup.loc[(row.model_b,row.seed),metric],f'paired seed/{row.model_a}/{row.seed}/{metric}')
    return calc

def load_completed_run(run_dir,repository_root=None):
    root=Path(run_dir).resolve();evaluation=root/'evaluation'
    if not (evaluation/'complete.json').is_file():
        raise FileNotFoundError(f'Completed benchmark results are absent at {evaluation}. Run the separate authorized benchmark CLI first, or set RUN_DIR to an already completed run. This notebook never starts computation of models.')
    identity=read_json(root/'run_identity.json');choices=read_json(root/'choices_frozen.json')
    prepared=read_json(root/'prepared.json');receipt=read_json(evaluation/'complete.json')
    protocol=read_json(root/'protocol.json');schedule=read_json(root/'schedule.json')
    run_sha=identity.get('run_sha256')
    require(run_sha==canonical_digest({k:v for k,v in identity.items() if k!='run_sha256'}),'Run identity digest differs')
    require(all(record.get('run_sha256')==run_sha for record in (choices,prepared,receipt)),'Saved artifacts belong to different run identities')
    require(identity.get('protocol_hash')==canonical_digest(protocol),'Protocol hash differs')
    require(identity.get('schedule_hash')==canonical_digest(schedule),'Schedule hash differs')
    check_hash(root/'choices_frozen.json',receipt.get('choices_frozen_sha256'))
    require(choices.get('choices_including_checkpoints_and_router_weights_frozen') is True,'Selections are not frozen')
    require(choices.get('evaluation_labels_accessed') is False,'Frozen selection receipt lacks label-isolation declaration')
    require(receipt.get('evaluation_truth_opened_only_after_all_choices') is True,'Completed evaluation lacks its choice gate')
    require(set(receipt.get('seeds',[]))==set(SEEDS) and tuple(receipt.get('months',[]))==MONTHS,'Evaluation receipt seed/calendar differs')
    require(protocol.get('role_bounds')=={'train':['1983-01','2007-11'],'monitor':['2008-01','2010-11'],'validation':['2011-01','2016-11'],'evaluation':['2017-01','2024-12']},'SAFE role boundaries differ from this notebook version')
    files_checked,bytes_checked=verify_files(root,prepared.get('files'))
    n,size=verify_files(evaluation,receipt.get('files'));files_checked+=n;bytes_checked+=size
    for name in [*(s+'.csv' for s in REQUIRED_CSV),*(s+'.json' for s in REQUIRED_JSON)]:
        require(name in receipt['files'],f'Required completed result is not hash-bound: {name}')
    jobs=choices.get('all_fit_jobs',[]);fit_hashes=choices.get('fit_complete_hashes',{})
    require(len(jobs)==len(fit_hashes) and {j.get('job_sha256') for j in jobs}==set(fit_hashes),'Frozen fit job/hash cohort differs')
    fit_receipts={}
    for job in jobs:
        key=job['job_sha256'];require(re.fullmatch('[0-9a-f]{64}',key) is not None,'Invalid fit identity')
        payload={k:v for k,v in job.items() if k not in ('root','job_sha256')}
        require(canonical_digest({'run_sha256':run_sha,'payload':payload})==key,'Fit job digest differs')
        path=root/'fits'/key[:16];check_hash(path/'complete.json',fit_hashes[key])
        saved=read_json(path/'complete.json')
        require(without_runtime_root(saved.get('job'))==without_runtime_root(job),'Fit configuration differs from frozen job')
        require(saved.get('evaluation_labels_accessed') is False,'Fit receipt lacks label-isolation declaration')
        n,size=verify_files(path,saved.get('files'));files_checked+=n+1;bytes_checked+=size+(path/'complete.json').stat().st_size
        fit_receipts[key]=saved
    require(all(o.get('job_sha256') in fit_hashes for o in choices.get('outputs',[])),'Output points to an unfrozen job')
    require(choices.get('baseline_job_sha256') in fit_hashes,'Baseline job is not frozen')
    if repository_root is not None:
        repo=Path(repository_root).resolve()
        for name,expected in identity.get('code',{}).items():
            require(re.fullmatch('[A-Za-z_][A-Za-z_0-9]*',name) is not None,'Invalid module name in identity')
            check_hash(repo/'caadapfl'/(name+'.py'),expected)
    frames={name:pd.read_csv(evaluation/(name+'.csv'),dtype={'target_month':str,'model':str,'client':str}) for name in REQUIRED_CSV}
    metadata={name:read_json(evaluation/(name+'.json')) for name in REQUIRED_JSON}
    metadata.update(identity=identity,choices=choices,prepared=prepared,evaluation=receipt,protocol=protocol,schedule=schedule)
    require(metadata['model_identities']==choices['outputs'],'Exported model identities differ from frozen outputs')
    keys=set(fit_hashes)
    for name in ('fit_timing','fit_cost_ledger'):
        frame=frames[name];require(not frame.job_sha256.duplicated().any() and set(frame.job_sha256)==keys,f'{name} does not count each unique frozen fit once')
    cost=frames['fit_cost_ledger']
    close(metadata['fit_cost_receipt']['sum_worker_wall_seconds'],cost.elapsed_wall_seconds.sum(),'fit wall-time ledger')
    calc=validate_tables(frames,metadata)
    return dict(root=root,evaluation=evaluation,metadata=metadata,frames=frames,
                calculated_macro=calc,fit_receipts=fit_receipts,
                integrity=dict(run_sha256=run_sha,files_verified=files_checked,bytes_verified=bytes_checked,
                               prediction_rows=len(frames['raw_predictions']),metric_reconciliation='R²/RMSE and paired points independently recomputed; no model execution'))

saved = load_completed_run(RUN_DIR, repository_root=REPOSITORY_ROOT)
tables = saved['frames']
records = saved['metadata']
FIGURE_PREFIX = 'SYNTHETIC FIXTURE | ' if records['evaluation'].get('fixture') else ''
print(json.dumps(saved['integrity'], indent=2))
print('Protocol:', records['protocol']['version'])
print('Frozen LR / mu / selected coefficients:', records['choices'].get('lr'),
      records['choices'].get('mu'), records['choices'].get('coefficients'))
print('Fixed component-parent coefficients:', records['choices'].get('ablation_coefficients'))
print('Effective feature aliases:', records['choices'].get('effective_group_aliases'))


## 3. Primary accuracy, all seeds, identities and checkpoints

Both equal-client macro R² and macro RMSE are primary reported accuracy metrics;
only R² chose candidates/checkpoints on outer validation. Pooled R²/RMSE are
separately named diagnostics. NSE and residual R² have the same formula.
The points below retain all five paired seeds and their arithmetic mean.
The display order is fixed in the parameter cell, not sorted by evaluation skill.
TRAIN_SELECTED aliases, if inspected, are duplicate evidence when their source
group matches an existing group.

In [ ]:
available = set(tables['seed_summary']['model'])
missing = set(VIEW_MODELS) - available
if missing:
    raise ValueError('Requested saved model labels are absent: ' + ', '.join(sorted(missing)))
macro = tables['aggregate_per_seed'].query("scope == 'macro_equal_client'")
view = macro[macro.model.isin(VIEW_MODELS)].copy()
summary = tables['seed_summary'].set_index('model').loc[list(VIEW_MODELS)]
print(summary[['R2_mean', 'R2_std', 'RMSE_mm_mean', 'RMSE_mm_std']].to_string())
print('\nAll paired seed R² values:')
print(view.pivot(index='model', columns='seed', values='R2').reindex(VIEW_MODELS)[list(SEEDS)].to_string())
identities = pd.DataFrame(records['model_identities'])
identity_columns = [c for c in ('label', 'kind', 'identity', 'scenario', 'group', 'alias_of_effective_group') if c in identities]
print('\nDeclared output identities:')
print(identities[identities.label.isin(VIEW_MODELS)][identity_columns].drop_duplicates().to_string(index=False))
raw = tables['raw_predictions']
if 'checkpoint_round' in raw:
    print('\nSaved checkpoint rounds (trees/baselines have no neural checkpoint):')
    print(raw[raw.model.isin(VIEW_MODELS)].groupby(['model','seed']).checkpoint_round.first().unstack('seed').reindex(VIEW_MODELS).to_string())
fig, axes = plt.subplots(1, 2, figsize=(13, max(5, .44 * len(VIEW_MODELS))), sharey=True)
for axis, metric, label in zip(axes, ('R2','RMSE_mm'), ('Equal-client macro R²','Equal-client macro RMSE (mm)')):
    for j, seed in enumerate(SEEDS):
        values = view[view.seed == seed].set_index('model').loc[list(VIEW_MODELS), metric]
        axis.scatter(values, np.arange(len(VIEW_MODELS)) + (j-2)*.045, s=22, label=str(seed), alpha=.8)
    axis.scatter(summary[metric+'_mean'], np.arange(len(VIEW_MODELS)), marker='D', color='black', s=32, label='Seed mean')
    axis.set_xlabel(label); axis.grid(axis='x', alpha=.25)
axes[0].set_yticks(np.arange(len(VIEW_MODELS)), labels=VIEW_MODELS)
axes[0].invert_yaxis(); axes[1].legend(title='Paired seed', loc='best', fontsize=8)
fig.suptitle(FIGURE_PREFIX + 'Saved FULL results: separate model identities, five paired seeds')
fig.tight_layout(); plt.show()


## 4. Prespecified paired uncertainty

Saved deltas are **A minus B**: positive R² and negative RMSE favor A. The
estimand is mean per-seed macro skill, not a prediction-ensemble metric. Bootstrap
intervals use 2,000 paired replicates, common circular target-date blocks across
all clients/seeds/models, and paired seed-index resampling. Twelve months is
primary; 6 and 24 are declared sensitivities. The intervals are marginal 95%
intervals, not simultaneous familywise bands.

The main protocol has 31 pairs. Five FULL component pairs use the fixed
(0.3,0.3,0.4) anchor parent rather than selected-coefficient CAAA; those estimands
must stay distinct. Optional annual-block sign randomization assumes block-sign
symmetry. Eight blocks yield 256 assignments and minimum two-sided nonzero
p=0.0078125; Holm over 31 pairs cannot reject at 0.05 with that resolution.
Do not substitute a more favorable test after seeing evaluation results.

In [ ]:
declared_pairs = [tuple(pair) for pair in records['protocol']['comparisons']]
if PAIR not in declared_pairs:
    raise ValueError('PAIR must be one of the saved prespecified comparisons.')
intervals = tables['paired_block_bootstrap_summary']
paired_view = intervals[(intervals.model_a == PAIR[0]) & (intervals.model_b == PAIR[1])]
print('Frozen comparison-family size:', len(declared_pairs))
print(paired_view[['metric','delta_a_minus_b','CI95_low','CI95_high','block_length_months','role']].to_string(index=False))
per_seed_pair = tables['paired_per_seed_summary']
print(per_seed_pair[(per_seed_pair.model_a == PAIR[0]) & (per_seed_pair.model_b == PAIR[1])].to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3))
for axis, metric, label in zip(axes, ('macro_R2','macro_RMSE_mm'), ('Δ macro R² (A−B; positive favors A)','Δ macro RMSE mm (A−B; negative favors A)')):
    rows = paired_view[paired_view.metric == metric].set_index('block_length_months').loc[[12,6,24]]
    axis.hlines(np.arange(3), rows.CI95_low, rows.CI95_high, color='tab:blue', linewidth=2)
    axis.scatter(rows.delta_a_minus_b, np.arange(3), color='black', zorder=3)
    axis.axvline(0, color='gray', linestyle='--', linewidth=1)
    axis.set_yticks(range(3), labels=('12 months: primary','6 months: sensitivity','24 months: sensitivity'))
    axis.invert_yaxis(); axis.set_xlabel(label); axis.grid(axis='x', alpha=.2)
fig.suptitle(FIGURE_PREFIX + PAIR[0] + ' versus ' + PAIR[1]); fig.tight_layout(); plt.show()
if records['evaluation'].get('pvalues'):
    tests = pd.DataFrame(read_json(saved['evaluation'] / 'block_sign_randomization_Holm.json'))
    print(tests[(tests.model_a == PAIR[0]) & (tests.model_b == PAIR[1])].to_string(index=False))
else:
    print('Optional sign-test policy was disabled for this completed evaluation.')


## 5. Saved monthly forecasts, event support and raw negatives

This plot uses a fixed client, a single declared seed and saved predictions.
It does not refit or average seed predictions. Trained heavy-event heads use
probability≥0.5; controls without a trained head use predicted rainfall greater
than training q80 and are labelled continuous rainfall scores. Read the saved
score-kind/decision columns when comparing events. Tail support is essential.
Monthly regional means do not establish flood-warning performance. Raw negative
forecasts are retained in all reported regression metrics.

In [ ]:
if PLOT_CLIENT not in CLIENTS or PLOT_SEED not in SEEDS:
    raise ValueError('PLOT_CLIENT/PLOT_SEED must belong to the frozen cohort.')
line_data = raw[(raw.client == PLOT_CLIENT) & (raw.seed == PLOT_SEED) &
                (raw.target_month >= PLOT_START) & raw.model.isin(PAIR)].copy()
if line_data.empty:
    raise ValueError('The explicit plot date window contains no saved rows.')
fig, axis = plt.subplots(figsize=(11, 4))
for model, part in line_data.groupby('model', sort=False):
    part = part.sort_values('target_month')
    dates = pd.to_datetime(part.target_month)
    axis.plot(dates, part.prediction, label=model, linewidth=1.6)
truth_line = line_data[line_data.model == PAIR[0]].sort_values('target_month')
axis.plot(pd.to_datetime(truth_line.target_month), truth_line.y, label='Observed CHIRPS v2', color='black', linewidth=1.7)
axis.axhline(0, color='gray', linewidth=.7); axis.set_ylabel('Monthly regional rainfall (mm)')
axis.set_title(FIGURE_PREFIX + PLOT_CLIENT + ', saved seed ' + str(PLOT_SEED))
axis.legend(fontsize=8); axis.grid(alpha=.2); fig.tight_layout(); plt.show()
event_cols = ['model','seed','client','actual_heavy_n','heavy_TP','heavy_FP','heavy_FN','heavy_TN',
              'heavy_precision','heavy_recall','heavy_F1','heavy_average_precision',
              'heavy_ROC_AUC','tail80_RMSE_mm','tail80_MAE_mm','heavy_score_kind','heavy_decision_rule']
event_rows = tables['per_client']
print(event_rows[(event_rows.model.isin(PAIR)) & (event_rows.seed == PLOT_SEED)][event_cols].to_string(index=False))
negatives = tables['negative_forecast_diagnostics']
print('\nRaw negative-forecast diagnostics:')
print(negatives[(negatives.model.isin(PAIR)) & (negatives.seed == PLOT_SEED)].to_string(index=False))


## 6. Work, communication and hardware

Unique completed fits count once, even when a development candidate is reused
or TRAIN_SELECTED produces an alias. Candidate work, final fits and router OOF
work must remain visible. Ditto's second stream is additional. Summed worker
wall times are not the elapsed wall time of a parallel schedule; worker CPU time
is a different quantity. The three-group selector has nine declared Ridge fits;
its timing is available only if separately instrumented.

Model/scaler/target-summary payload bytes omit transport, serialization and
encryption overhead. Graph delivery aliases are not added twice. Shared setup
artifacts are counted once for the graph family. Central numeric array access
is a disclosure estimate, not measured network transfer. Updates and summaries
are plaintext; secure aggregation and differential privacy are not implemented.
The enriched graph has privileged offline aligned-rainfall access.

In [ ]:
cost = tables['fit_cost_ledger']
cost_columns = ['elapsed_wall_seconds','worker_process_cpu_seconds','optimizer_steps',
                'training_examples_presented','message_uplink_bytes','message_downlink_bytes',
                'graph_matrix_setup_delivery_alias_bytes','input_scaler_setup_bytes',
                'target_decode_summary_bytes','centralized_training_array_access_bytes']
cost_columns = [name for name in cost_columns if name in cost]
print(cost.groupby('kind')[cost_columns].sum(min_count=1).to_string())
print('\nUnique complete fits and developmental/final reuse:')
print(cost.groupby(['kind','stage_roles']).size().rename('unique_fits').to_string())
print('\nCost interpretation receipt:')
print(json.dumps(records['fit_cost_receipt'], indent=2))
graph_setup = records['shared_graph_setup_ledger']
print('\nShared graph setup payload bytes:', sum(int(message['bytes']) for message in graph_setup['messages']))
print(json.dumps({k:v for k,v in graph_setup.items() if k != 'messages'}, indent=2))
print('\nHardware/reference versions:')
print(json.dumps(records['hardware'], indent=2))


## 7. Interpretation and source receipts

Inspect the frozen selected settings, zero coefficients, exact model identities
and per-client results before writing a conclusion. Fixed-anchor component
effects do not prove that a zero-weight component in the selected primary is
active. The separately named static classical router uses overall/chronological
last-half R²; it does not reproduce the manuscript's underspecified tail-DPRC
equation or implement a climate gate.

Compare skill with explicit centralized/local/parameter-federated controls,
cost and source assumptions. Graph/climate/router complexity with unsupported
benefit may be simplified; no method is presumed best on every metric.

Read `docs/FROZEN_PROTOCOL.md`, `data/manifest.json`, `provenance/validation/`,
`provenance/source_tables/` and `ENVIRONMENT.json` alongside this run. Source
spot checks have limited temporal scope; full-period gates and any official
provider recovery must be explicit. Optional IMD, Solar and GLDAS results need
separate matched cohorts. Archive fixtures are provenance rather than runnable
public tests. This notebook creates no new model result and publishes nothing.

In [ ]:
print('Run identity:', records['identity']['run_sha256'])
print('Inactive selected-primary aggregation terms:',
      records['choices'].get('selected_primary_zero_aggregation_components', []))
print('Saved support:', len(CLIENTS), 'clients ×', len(MONTHS), 'months ×', len(SEEDS), 'paired seeds per model')
print('Source/code checksums recorded in run identity:')
print(json.dumps({'source': records['identity'].get('source'), 'code': records['identity'].get('code')}, indent=2))
print('This companion inspected saved artifacts only; no model execution occurred.')
